## Section 1 - Supervised Learning


In [1]:
# Shared setup and utility functions
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

warnings.simplefilter("ignore", pd.errors.PerformanceWarning)

try:
    PROJECT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    _cwd = Path.cwd().resolve()
    PROJECT_DIR = _cwd.parent if _cwd.name.lower() == "code" else _cwd

DATA_DIR = PROJECT_DIR / "data"
SUPERVISED_DIR = DATA_DIR / "supervised"
OUTPUT_DIR = SUPERVISED_DIR
PLOT_DIR = PROJECT_DIR / "plots" / "supervised"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = DATA_DIR / "H20121242Data.csv"
RAW_CODEBOOK_PATH = DATA_DIR / "H20121242Codebook.xls"
LABELED_PATH = SUPERVISED_DIR / "supervised_labeled.csv"
FEATURE_DICTIONARY_PATH = SUPERVISED_DIR / "feature_dictionary.csv"


def safe_to_csv(frame, path, index=False):
    try:
        frame.to_csv(path, index=index, encoding="utf-8")
    except PermissionError:
        print(f"Could not overwrite {path}; it may be open in another program. Continuing.")


def f1_score_binary(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return 0.0 if denom == 0 else float(2 * tp / denom)


### Q1 - Preprocessing


In [2]:
# The raw CBS file is Windows Hebrew encoded. Outputs are written as UTF-8.
df = pd.read_csv(RAW_DATA_PATH, encoding="cp1255")
assert df.shape == (1214, 134), f"Unexpected raw shape: {df.shape}"

features = pd.DataFrame(index=df.index)
feature_rows = []
handled_sources = set()
idk_feature_names = []
score_feature_names = []

LEAKAGE_COLUMNS = {"Hovot", "Hovot2", "Hovot3", "Hovot4", "Hovot5", "Hovot6", "Hovot7", "Hovot8", "Hovot9", "HovotLechatzim"}
TEXT_COLUMNS = ["OverVaI7", "Halvaa10", "Hovot9", "Hashkaot10", "HipusM9", "MekorotM10"]
TEXT_VALID_CODE_RANGES = {
    "OverVaI7": {1, 2},
    "Halvaa10": {1, 2, 3},
    "Hovot9": {1, 2, 3, 4, 5},
    "Hashkaot10": {1, 2, 3},
    "HipusM9": {1, 2, 3, 4, 5},
    "MekorotM10": {1, 2, 3},
}
NON_FEATURE_COLUMNS = {"SerialNumber", "Shana", "nn"}

def safe_name(value):
    value = str(value).strip()
    value = value.replace("+", "plus")
    value = re.sub(r"[^0-9A-Za-z_]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "missing"

def numeric_code_series(source):
    return pd.to_numeric(df[source], errors="coerce")

def register_feature(name, source_columns, transformation, feature_type, notes=""):
    if isinstance(source_columns, str):
        source_columns = [source_columns]
    feature_rows.append({
        "feature_name": name,
        "source_columns": ",".join(source_columns),
        "feature_type": feature_type,
        "transformation": transformation,
        "notes": notes,
    })
    handled_sources.update(source_columns)

def add_feature(name, values, source_columns, transformation, feature_type="numeric", notes="", is_score=False):
    features[name] = pd.to_numeric(values, errors="coerce")
    register_feature(name, source_columns, transformation, feature_type, notes)
    if is_score:
        score_feature_names.append(name)

def add_flag(name, values, source_columns, transformation, notes=""):
    features[name] = pd.Series(values, index=df.index).fillna(False).astype(int)
    register_feature(name, source_columns, transformation, "indicator", notes)

def add_one_hot(source, prefix=None, transformation="one-hot categorical encoding", labels=None, include_missing=True, notes=""):
    prefix = prefix or source
    s = numeric_code_series(source)
    cats = pd.Series(index=df.index, dtype="object")
    for idx, val in s.items():
        if pd.isna(val):
            cats.loc[idx] = "missing" if include_missing else np.nan
        else:
            key = int(val) if float(val).is_integer() else val
            label = labels.get(key, f"code_{safe_name(key)}") if labels else f"code_{safe_name(key)}"
            cats.loc[idx] = safe_name(label)
    dummies = pd.get_dummies(cats, prefix=prefix, dtype=int)
    for col in dummies.columns:
        features[col] = dummies[col]
        register_feature(col, source, transformation, "indicator", notes)

def add_mapped_score(source, name, mapping, transformation, idk_codes=None, not_relevant_codes=None, notes=""):
    s = numeric_code_series(source)
    score = s.map(mapping)
    add_feature(name, score, source, transformation, "semantic_score", notes, is_score=True)
    add_flag(f"{name}_missing", s.isna(), source, f"missing indicator for {source}")
    unmapped = s.notna() & score.isna()
    if unmapped.any():
        add_flag(f"{name}_unmapped", unmapped, source, f"non-missing value not found in score map for {source}")
    if idk_codes:
        flag_name = f"{name}_idk"
        add_flag(flag_name, s.isin(idk_codes), source, f"IDK indicator for {source}")
        idk_feature_names.append(flag_name)
    if not_relevant_codes:
        add_flag(f"{name}_not_relevant", s.isin(not_relevant_codes), source, f"not-relevant indicator for {source}")

def add_binary_yes_no(source, prefix=None, notes="1=yes, 2=no"):
    prefix = prefix or source
    s = numeric_code_series(source)
    add_feature(f"{prefix}_yes", s.map({1: 1, 2: 0}), source, "binary yes/no score", "semantic_score", notes, is_score=True)
    add_flag(f"{prefix}_missing", s.isna(), source, f"missing indicator for {source}")

def add_yes_no_idk_indicators(source, prefix=None, notes="1=yes, 2=no, 3=IDK"):
    prefix = prefix or source
    s = numeric_code_series(source)
    add_flag(f"{prefix}_yes", s.eq(1), source, "yes indicator", notes)
    add_flag(f"{prefix}_no", s.eq(2), source, "no indicator", notes)
    idk_name = f"{prefix}_idk"
    add_flag(idk_name, s.eq(3), source, "IDK indicator", notes)
    add_flag(f"{prefix}_missing", s.isna(), source, "missing indicator", notes)
    idk_feature_names.append(idk_name)

def add_open_bin_score(source, name, mapping, top_codes, unknown_codes=None, transformation="mapped binned numeric value"):
    s = numeric_code_series(source)
    unknown_codes = set(unknown_codes or [])
    clean = s.mask(s.isin(unknown_codes))
    add_feature(name, clean.map(mapping), source, transformation, "numeric", "open-ended bins use extrapolated midpoint", is_score=True)
    add_flag(f"{name}_missing", s.isna(), source, f"missing indicator for {source}")
    add_flag(f"{name}_top_bin", s.isin(top_codes), source, f"top/open bin indicator for {source}")
    for code in unknown_codes:
        add_flag(f"{name}_code_{int(code)}", s.eq(code), source, f"special code {int(code)} indicator for {source}")

# -----------------------------
# Label and metadata
# -----------------------------
y = df["Hovot"].map({1: 0, 2: 1})
labeled_mask = y.notna()
unlabeled_mask = y.isna()
# based on manual checks on the original given data
assert int(labeled_mask.sum()) == 1153, int(labeled_mask.sum())
assert int((y == 0).sum()) == 923, int((y == 0).sum())
assert int((y == 1).sum()) == 230, int((y == 1).sum())
assert int(unlabeled_mask.sum()) == 61, int(unlabeled_mask.sum())

# -----------------------------
# Profile metrics
# -----------------------------
add_one_hot("Machoz", "district")
add_open_bin_score("Ms_Nefashot", "household_size_mid", {1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7.5}, top_codes=[7])
add_one_hot("Minn", "gender")
add_open_bin_score("Gil", "age_mid", {1: 23, 2: 28, 3: 33, 4: 38, 5: 43, 6: 48, 7: 53, 8: 58, 9: 63, 10: 70, 11: 80}, top_codes=[11])
add_one_hot("MatzavMishp", "family_status")
add_one_hot("Pop_Group", "population_group")
add_one_hot("DatSafa_C", "religion_language")
add_one_hot("HesderDiyur", "living_arrangement")
add_open_bin_score("ShnotLimudKlali_C", "education_years_mid", {1: 2.5, 2: 6.5, 3: 9.5, 4: 11.5, 5: 14, 6: 17}, top_codes=[6])
add_one_hot("MatzavTaasuka_C", "employment_status")
add_open_bin_score("HachnasaKolelet", "household_income_mid", {1: 1250, 2: 3250.5, 3: 4500.5, 4: 5750.5, 5: 7250.5, 6: 9000.5, 7: 11500.5, 8: 15000.5, 9: 20500.5, 10: 27500.5, 11: 0}, top_codes=[10], unknown_codes=[98, 99])
add_open_bin_score("HachnasaAvoda", "work_income_mid", {1: 1000, 2: 2500.5, 3: 3500.5, 4: 4500.5, 5: 5500.5, 6: 6750.5, 7: 8750.5, 8: 12000.5, 9: 17500.5, 10: 24500.5, 11: 0}, top_codes=[10], unknown_codes=[98, 99])
add_one_hot("SemelMishlach_wp", "profession")

# -----------------------------
# Quiz metrics
# -----------------------------
# Q1/Q3/Q11/Q26 style yes/no answers.
for col in ["OverVashav", "OverVaI1", "OverVaI2", "OverVaI3", "OverVaI4", "OverVaI5", "OverVaI6", "Mashkanta1", "NoseCaspiHarhavaNihul", "NoseCaspiHarhavaCarhanut", "NoseCaspiHarhavaHalvaot", "NoseCaspiHarhavaHishonot", "NoseCaspiHarhavaBituhim", "NoseCaspiHarhavaMashkanta", "NoseCaspiHarhavaEynCoreh"]:
    add_binary_yes_no(col)

# Q2 bank balance tracking: 2 > 3 > 4 > 5 > 1.
add_mapped_score("OverVaMaakav", "bank_balance_tracking_score", {2: 4, 3: 3, 4: 2, 5: 1, 1: 0}, "ordinal score: weekly+ tracking highest, no tracking lowest")

# Q4/Q8/Q10/Q13/Q16/Q19 yes/no/IDK style fields.
for col in ["OverVaAcher", "KartisAshP1", "KartisAshP2", "KartisAshP3", "KartisAshP4", "KartisAshS1", "KartisAshS2", "KartisAshS3", "KartisAshS4", "KartisAshS5", "KartisAshS6", "KartisAshS7", "KartisAshNechsam", "Halvaa2", "Halvaa3", "Halvaa4", "Halvaa5", "Halvaa6", "Halvaa7", "Halvaa8", "Halvaa9", "Hashkaot1", "Hashkaot2", "Hashkaot3", "Hashkaot4", "Hashkaot5", "Hashkaot6", "Hashkaot7", "Hashkaot8", "Hashkaot9", "SherutY1", "SherutY2", "SherutY3", "SherutY4", "SherutY5", "SherutY6"]:
    add_yes_no_idk_indicators(col)

# Q5 banking events: 1 good, 5 bad.
for col in ["OverVaEruim1", "OverVaEruim2", "OverVaEruim3"]:
    add_mapped_score(col, f"{col}_frequency_good_behavior_score", {1: 4, 2: 3, 3: 2, 4: 1, 5: 0}, "frequency score: never is best, usual state is worst")

# Q6/Q17 decision maker: categorical, with IDK as explicit signal.
for col in ["HeshbonMaakav", "HashkaotMaclit"]:
    add_one_hot(col, col, "one-hot decision-maker category")
    s = numeric_code_series(col)
    flag_name = f"{col}_idk"
    add_flag(flag_name, s.eq(8), col, "decision-maker IDK indicator")
    idk_feature_names.append(flag_name)

# Q7 credit cards: 1->0, ..., 5->4+ with extrapolated top value.
add_open_bin_score("KartisAshKama", "credit_cards_count", {1: 0, 2: 1, 3: 2, 4: 3, 5: 4.5}, top_codes=[5], transformation="credit-card count from coded answers")

# Q12 mortgage information sources: categorical, with IDK and structural no-mortgage signal.
add_one_hot("MashkantaMekorot", "mortgage_info_source", "one-hot mortgage information source")
add_flag("mortgage_info_source_idk", numeric_code_series("MashkantaMekorot").eq(7), "MashkantaMekorot", "mortgage-source IDK indicator")
idk_feature_names.append("mortgage_info_source_idk")
add_flag("mortgage_info_source_not_app_no_mortgage", numeric_code_series("Mashkanta1").eq(2) & numeric_code_series("MashkantaMekorot").isna(), ["Mashkanta1", "MashkantaMekorot"], "structural missingness: no mortgage, so source not applicable")

# Q13 overall loan yes/no.
add_binary_yes_no("Halvaa", "has_loan")

# Q18 insurance: IDK is intentionally represented as a strong signal.
for col in ["Bituhim1", "Bituhim2", "Bituhim3", "Bituhim4", "Bituhim5", "Bituhim6"]:
    add_one_hot(col, col, "one-hot insurance answer: yes/no/IDK/no need")
    s = numeric_code_series(col)
    idk_name = f"{col}_idk_strong_signal"
    add_flag(idk_name, s.eq(3), col, "IDK is a strong insurance uncertainty signal")
    idk_feature_names.append(idk_name)

# Q20 financial knowledge: all correct answers are yes. yes > IDK > no.
for col in ["NosCas1", "NosCas2", "NosCas3", "NosCas4"]:
    add_mapped_score(col, f"{col}_knowledge_score", {1: 2, 3: 1, 2: 0}, "financial knowledge score: yes correct, IDK middle, no lowest", idk_codes=[3])

# Q21 interest calculation numeric/free answer.
s21 = numeric_code_series("HishuvRibit")
q21_score = pd.Series(np.nan, index=df.index, dtype="float64")
q21_score[s21.eq(1020)] = 4
q21_score[(s21 >= 1000) & (s21 <= 1500) & ~s21.eq(1020)] = 3
q21_score[s21.eq(2)] = 2
q21_score[s21 > 1500] = 1
q21_score[(s21 < 1000) & ~s21.eq(2)] = 0
add_feature("interest_calc_1_score", q21_score, "HishuvRibit", "Q21 score: 1020 highest; 1000-1500 good; IDK middle; >1500 low; <1000 lowest", "semantic_score", is_score=True)
add_flag("interest_calc_1_idk", s21.eq(2), "HishuvRibit", "Q21 IDK code indicator")
add_flag("interest_calc_1_missing", s21.isna(), "HishuvRibit", "Q21 missing indicator")
idk_feature_names.append("interest_calc_1_idk")

# Q22/Q23 financial knowledge scores.
add_mapped_score("HishuvRibitDribit", "interest_calc_2_score", {1: 5, 2: 4, 3: 3, 5: 2, 4: 1}, "Q22 score order: 1 > 2 > 3 > 5 > 4", idk_codes=[5])
add_mapped_score("ShiurRibitBank", "saving_program_interest_score", {2: 4, 3: 3, 1: 2, 4: 1}, "Q23 score order: 2 > 3 > 1 > 4", idk_codes=[4])

# Q24 financial information search frequency: higher is better.
for col in ["HipusM1", "HipusM2", "HipusM3", "HipusM4", "HipusM5", "HipusM6", "HipusM7", "HipusM8"]:
    add_mapped_score(col, f"{col}_info_search_score", {1: 0, 2: 1, 3: 2, 4: 3, 5: 4}, "Q24 information-search frequency score: higher is better")

# Q25 agreement statements: positive and negative directions, code 5 not relevant.
positive_q25 = [1, 3, 5, 6, 7, 8, 10]
negative_q25 = [2, 4, 9]
for i in positive_q25:
    add_mapped_score(f"NoseCaspiM{i}", f"NoseCaspiM{i}_attitude_score", {1: 4, 2: 3, 3: 2, 4: 1, 5: 0}, "Q25 positive statement score: strongly agree highest; not relevant zero", not_relevant_codes=[5])
for i in negative_q25:
    add_mapped_score(f"NoseCaspiM{i}", f"NoseCaspiM{i}_attitude_score", {1: 1, 2: 2, 3: 3, 4: 4, 5: 0}, "Q25 negative statement score: strongly disagree highest; not relevant zero", not_relevant_codes=[5])

# Q26 aggregate: 1-6 interests equal, 7 means no perceived need.
interest_cols = ["NoseCaspiHarhavaNihul", "NoseCaspiHarhavaCarhanut", "NoseCaspiHarhavaHalvaot", "NoseCaspiHarhavaHishonot", "NoseCaspiHarhavaBituhim", "NoseCaspiHarhavaMashkanta"]
interest_yes = pd.DataFrame({col: numeric_code_series(col).eq(1).astype(int) for col in interest_cols})
add_feature("knowledge_expansion_topics_count", interest_yes.sum(axis=1), interest_cols, "count of selected knowledge-expansion topics among fields 1-6", "aggregate_count")
add_flag("knowledge_expansion_no_need", numeric_code_series("NoseCaspiHarhavaEynCoreh").eq(1), "NoseCaspiHarhavaEynCoreh", "selected no need to expand financial knowledge")

# Q27 information-source interest: non-ordinal indicators.
for col in ["MekorotM1", "MekorotM2", "MekorotM3", "MekorotM4", "MekorotM5", "MekorotM6", "MekorotM7", "MekorotM8", "MekorotM9"]:
    add_one_hot(col, col, "Q27 non-ordinal source-interest one-hot")
    s = numeric_code_series(col)
    idk_name = f"{col}_idk"
    add_flag(idk_name, s.eq(3), col, "Q27 source-interest IDK indicator")
    idk_feature_names.append(idk_name)

# Q28 preferred source: categorical fallback.
add_one_hot("MakorMuadaf", "preferred_info_source", "Q28 preferred information source one-hot")
add_one_hot("MakorMahooM", "preferred_source_number", "Q28 selected source number one-hot")

# Q29 mortgage knowledge: 1 highest, 5 lowest.
add_mapped_score("RamatYedaMash", "mortgage_knowledge_score", {1: 4, 2: 3, 3: 2, 4: 1, 5: 0}, "Q29 mortgage knowledge score: 1 highest, 5 lowest")

# Aggregate IDK and score features.
if idk_feature_names:
    idk_matrix = features[idk_feature_names]
    add_feature("idk_total", idk_matrix.sum(axis=1), idk_feature_names, "total number of IDK indicators across engineered features", "aggregate_count")
    add_feature("idk_rate", idk_matrix.mean(axis=1), idk_feature_names, "share of IDK indicators across engineered features", "aggregate_rate")

def add_mean_feature(name, cols, notes):
    existing = [c for c in cols if c in features.columns]
    if existing:
        add_feature(name, features[existing].mean(axis=1, skipna=True), existing, "row-wise mean of related semantic scores", "aggregate_score", notes, is_score=True)

add_mean_feature("q20_knowledge_mean", [f"NosCas{i}_knowledge_score" for i in range(1, 5)], "Q20 financial knowledge mean")
add_mean_feature("q24_info_search_mean", [f"HipusM{i}_info_search_score" for i in range(1, 9)], "Q24 information search mean")
add_mean_feature("q25_attitude_mean", [f"NoseCaspiM{i}_attitude_score" for i in range(1, 11)], "Q25 financial attitude mean")
add_mean_feature("financial_knowledge_mean", ["q20_knowledge_mean", "interest_calc_1_score", "interest_calc_2_score", "saving_program_interest_score", "mortgage_knowledge_score"], "combined financial knowledge proxy")

# Safety net: any remaining non-excluded source columns become one-hot categorical features.
excluded_sources = LEAKAGE_COLUMNS | set(TEXT_COLUMNS) | NON_FEATURE_COLUMNS
remaining = [c for c in df.columns if c not in handled_sources and c not in excluded_sources]
for col in remaining:
    add_one_hot(col, col, "fallback one-hot for remaining non-excluded source column", notes="Added so no usable raw column is silently ignored.")

# Validate leakage and text exclusions by feature-dictionary sources.
feature_dict = pd.DataFrame(feature_rows)
used_source_tokens = set()
for sources in feature_dict["source_columns"]:
    used_source_tokens.update([s for s in str(sources).split(",") if s])
assert not (used_source_tokens & LEAKAGE_COLUMNS), f"Leakage sources used: {used_source_tokens & LEAKAGE_COLUMNS}"
assert not (used_source_tokens & set(TEXT_COLUMNS)), f"Text sources used in features: {used_source_tokens & set(TEXT_COLUMNS)}"

# Neutral-fill remaining NaNs in engineered numeric features using labeled-row medians.
filled_features = features.copy()
fill_values = filled_features.loc[labeled_mask].median(numeric_only=True).fillna(0)
filled_features = filled_features.fillna(fill_values).fillna(0)
assert not filled_features.isna().any().any(), "Feature matrix still contains NaN after neutral fill"

# Build outputs.
labeled_out = pd.concat([
    df.loc[labeled_mask, ["SerialNumber"]].reset_index(drop=True),
    y.loc[labeled_mask].astype(int).rename("y_debt").reset_index(drop=True),
    df.loc[labeled_mask, "nn"].rename("sample_weight").reset_index(drop=True),
    filled_features.loc[labeled_mask].reset_index(drop=True),
], axis=1)

unlabeled_out = pd.concat([
    df.loc[unlabeled_mask, ["SerialNumber"]].reset_index(drop=True),
    df.loc[unlabeled_mask, "Hovot"].rename("raw_Hovot").reset_index(drop=True),
    df.loc[unlabeled_mask, "nn"].rename("sample_weight").reset_index(drop=True),
    filled_features.loc[unlabeled_mask].reset_index(drop=True),
], axis=1)

text_df = df[TEXT_COLUMNS].copy()
text_cleanup_records = []
for col, valid_codes in TEXT_VALID_CODE_RANGES.items():
    stripped = text_df[col].dropna().astype(str).str.strip()
    numeric_code = stripped.str.fullmatch(r"\d+")
    numeric_values = pd.to_numeric(stripped.where(numeric_code), errors="coerce")
    valid_numeric_code = numeric_code & numeric_values.isin(valid_codes)
    for idx, value in stripped[valid_numeric_code].items():
        text_cleanup_records.append({
            "SerialNumber": df.loc[idx, "SerialNumber"],
            "source_column": col,
            "removed_value": value,
            "reason": "single numeric answer code in free-text detail field",
        })
    text_df.loc[stripped[valid_numeric_code].index, col] = np.nan

text_records = []
for col in TEXT_COLUMNS:
    non_missing = text_df[col].notna()
    for idx, value in text_df.loc[non_missing, col].items():
        text_records.append({
            "SerialNumber": df.loc[idx, "SerialNumber"],
            "source_column": col,
            "raw_value": value,
        })
free_text_audit = pd.DataFrame(text_records, columns=["SerialNumber", "source_column", "raw_value"])
free_text_numeric_code_cleanup = pd.DataFrame(text_cleanup_records, columns=["SerialNumber", "source_column", "removed_value", "reason"])

# Final checks.
assert labeled_out.shape[0] == 1153
assert int(labeled_out["y_debt"].eq(0).sum()) == 923
assert int(labeled_out["y_debt"].eq(1).sum()) == 230
assert unlabeled_out.shape[0] == 61
assert "Hovot" not in labeled_out.columns
for leakage_col in LEAKAGE_COLUMNS:
    assert leakage_col not in labeled_out.columns, leakage_col
for text_col in TEXT_COLUMNS:
    assert text_col not in labeled_out.columns, text_col

# Export UTF-8 CSVs.
labeled_path = OUTPUT_DIR / "supervised_labeled.csv"
unlabeled_path = OUTPUT_DIR / "supervised_unlabeled_hovot_missing.csv"
text_path = OUTPUT_DIR / "free_text_audit.csv"
text_cleanup_path = OUTPUT_DIR / "free_text_numeric_code_cleanup.csv"
dict_path = OUTPUT_DIR / "feature_dictionary.csv"

labeled_out.to_csv(labeled_path, index=False, encoding="utf-8")
unlabeled_out.to_csv(unlabeled_path, index=False, encoding="utf-8")
free_text_audit.to_csv(text_path, index=False, encoding="utf-8")
free_text_numeric_code_cleanup.to_csv(text_cleanup_path, index=False, encoding="utf-8")
feature_dict.to_csv(dict_path, index=False, encoding="utf-8")

print("Raw shape:", df.shape)
print("Labeled output shape:", labeled_out.shape)
print("Unlabeled output shape:", unlabeled_out.shape)
print("Debt label counts:")
print(labeled_out["y_debt"].value_counts().sort_index())
print("Free-text counts by column:")
print(free_text_audit["source_column"].value_counts().reindex(TEXT_COLUMNS, fill_value=0))
print("Rows with any free-text value:", free_text_audit["SerialNumber"].nunique())
print("Special income code counts:")
print(pd.DataFrame({
    "HachnasaKolelet_98": [int(numeric_code_series("HachnasaKolelet").eq(98).sum())],
    "HachnasaKolelet_99": [int(numeric_code_series("HachnasaKolelet").eq(99).sum())],
    "HachnasaAvoda_98": [int(numeric_code_series("HachnasaAvoda").eq(98).sum())],
    "HachnasaAvoda_99": [int(numeric_code_series("HachnasaAvoda").eq(99).sum())],
}))
print("Population group vs religion/language:")
print(pd.crosstab(df["Pop_Group"], df["DatSafa_C"], dropna=False))
print("Output directory:", OUTPUT_DIR)


Raw shape: (1214, 134)
Labeled output shape: (1153, 447)
Unlabeled output shape: (61, 447)
Debt label counts:
y_debt
0    923
1    230
Name: count, dtype: int64
Free-text counts by column:
source_column
OverVaI7       9
Halvaa10      28
Hovot9        18
Hashkaot10    18
HipusM9       16
MekorotM10     6
Name: count, dtype: int64
Rows with any free-text value: 81
Special income code counts:
   HachnasaKolelet_98  HachnasaKolelet_99  HachnasaAvoda_98  HachnasaAvoda_99
0                 139                  20                16                17
Population group vs religion/language:
DatSafa_C    1   2    3
Pop_Group              
1.0          0   1  995
2.0        162  16   39
NaN          0   0    1
Output directory: C:\Users\sutov\Desktop\Huji\Year 3\Sem B\Modern Statistics Data\FinalProject\data\supervised


### Q2 - Lasso Path


In [3]:
DATA_DIR = SUPERVISED_DIR
COEFFICIENTS_PATH = DATA_DIR / "lasso_path_coefficients.csv"
TOP_FEATURES_PATH = DATA_DIR / "lasso_path_top_features.csv"
PLOT_PNG_PATH = PLOT_DIR / "lasso_path.png"

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


def run_lasso_path_sklearn():
    data = pd.read_csv(LABELED_PATH)
    feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
    feature_names = [c for c in data.columns if c not in {"SerialNumber", "y_debt", "sample_weight"}]
    X_raw = data[feature_names].to_numpy(dtype=float)
    y = data["y_debt"].to_numpy(dtype=int)

    scaler = StandardScaler()
    X_scaled_all = scaler.fit_transform(X_raw)
    non_constant = scaler.scale_ > 1e-12
    X = X_scaled_all[:, non_constant]
    kept_features = np.array(feature_names, dtype=object)[non_constant]

    C_values = np.geomspace(0.05, 30.0, 35)
    coefs = []
    intercepts = []
    active_counts = []
    for C in C_values:
        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=float(C),
            max_iter=5000,
            class_weight=None,
            random_state=42,
        )
        model.fit(X, y)
        beta = model.coef_.ravel()
        coefs.append(beta)
        intercepts.append(float(model.intercept_[0]))
        active_counts.append(int(np.abs(beta).gt(1e-10).sum()) if hasattr(np.abs(beta), "gt") else int((np.abs(beta) > 1e-10).sum()))

    coef_path = np.vstack(coefs)
    active = np.abs(coef_path) > 1e-10
    entry_steps = np.where(active.any(axis=0), active.argmax(axis=0) + 1, np.inf)
    max_abs = np.abs(coef_path).max(axis=0)

    source_lookup = feature_dictionary.drop_duplicates("feature_name").set_index("feature_name")
    top_rows = []
    for j, feature in enumerate(kept_features):
        top_rows.append({
            "feature_name": feature,
            "max_abs_standardized_coef": float(max_abs[j]),
            "final_standardized_coef": float(coef_path[-1, j]),
            "entry_C": np.nan if not np.isfinite(entry_steps[j]) else float(C_values[int(entry_steps[j]) - 1]),
            "entry_step": np.nan if not np.isfinite(entry_steps[j]) else int(entry_steps[j]),
            "source_columns": source_lookup["source_columns"].get(feature, ""),
            "feature_type": source_lookup["feature_type"].get(feature, ""),
            "transformation": source_lookup["transformation"].get(feature, ""),
        })
    top_features = pd.DataFrame(top_rows).query("max_abs_standardized_coef > 0").sort_values(
        ["max_abs_standardized_coef", "entry_step"], ascending=[False, True]
    ).reset_index(drop=True)

    path_df = pd.DataFrame(coef_path, columns=kept_features)
    path_df.insert(0, "active_count", active_counts)
    path_df.insert(0, "intercept", intercepts)
    path_df.insert(0, "log_C", np.log(C_values))
    path_df.insert(0, "C", C_values)

    safe_to_csv(path_df, COEFFICIENTS_PATH)
    safe_to_csv(top_features, TOP_FEATURES_PATH)

    top_plot = top_features.head(15)["feature_name"].tolist()
    plot_idx = [list(kept_features).index(name) for name in top_plot]
    plt.figure(figsize=(13, 7))
    for idx, name in zip(plot_idx, top_plot):
        plt.plot(np.log(C_values), coef_path[:, idx], linewidth=2, label=name)
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("log(C), weaker regularization to the right")
    plt.ylabel("standardized coefficient")
    plt.title("L1 logistic-regression path for debt classification")
    plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOT_PNG_PATH, dpi=140)
    plt.close()

    print("Lasso path completed with sklearn LogisticRegression")
    print("Features used:", len(kept_features))
    print("Active coefficients at final C:", active_counts[-1])
    print(top_features.head(15)[["feature_name", "entry_step", "final_standardized_coef"]].to_string(index=False))
    return {"coefficients": path_df, "top_features": top_features}


In [4]:
lasso_results = run_lasso_path_sklearn() # uses our custom lasso path function
lasso_top_features = lasso_results['top_features']
lasso_top_features.head(20)


Lasso path completed with sklearn LogisticRegression
Features used: 444
Active coefficients at final C: 330
                           feature_name  entry_step  final_standardized_coef
               financial_knowledge_mean        20.0                36.761705
               employment_status_code_1        11.0               -28.853217
                work_income_mid_missing        21.0               -28.324068
                  interest_calc_2_score         7.0               -25.449236
                           SherutY3_idk         8.0               -23.393826
NoseCaspiM3_attitude_score_not_relevant         1.0               -20.623700
                           has_loan_yes         1.0               -20.323122
                      MekorotM4_missing         5.0                17.903132
                           SherutY4_idk         6.0                16.962623
                    KartisAshP4_missing        16.0               -16.116566
                        Halvaa2_missing      

,feature_name,max_abs_standardized_coef,final_standardized_coef,entry_C,entry_step,source_columns,feature_type,transformation
0,financial_knowledge_mean,36.761705,36.761705,1.784293,20.0,"q20_knowledge_mean,interest_calc_1_score,inter...",aggregate_score,row-wise mean of related semantic scores
1,employment_status_code_1,28.853217,-28.853217,0.328151,11.0,MatzavTaasuka_C,indicator,one-hot categorical encoding
2,work_income_mid_missing,28.324068,-28.324068,2.153657,21.0,HachnasaAvoda,indicator,missing indicator for HachnasaAvoda
3,interest_calc_2_score,25.449236,-25.449236,0.154608,7.0,HishuvRibitDribit,semantic_score,Q22 score order: 1 > 2 > 3 > 5 > 4
4,SherutY3_idk,23.393826,-23.393826,0.186613,8.0,SherutY3,indicator,IDK indicator
5,NoseCaspiM3_attitude_score_not_relevant,20.623700,-20.623700,0.050000,1.0,NoseCaspiM3,indicator,not-relevant indicator for NoseCaspiM3
6,has_loan_yes,20.323122,-20.323122,0.050000,1.0,Halvaa,semantic_score,binary yes/no score
7,MekorotM4_missing,17.903132,17.903132,0.106123,5.0,MekorotM4,indicator,Q27 non-ordinal source-interest one-hot
8,SherutY4_idk,16.962623,16.962623,0.128092,6.0,SherutY4,indicator,IDK indicator
9,KartisAshP4_missing,16.116566,-16.116566,0.840669,16.0,KartisAshP4,indicator,missing indicator


### Q3 - PCA


In [5]:
DATA_DIR = SUPERVISED_DIR
PCA_EXPLAINED_PATH = DATA_DIR / "pca_explained_variance.csv"
PCA_LOADINGS_PATH = DATA_DIR / "pca_top_loadings.csv"
PCA_PR_AUC_PATH = DATA_DIR / "pca_pr_auc.csv"
PCA_SCORES_PATH = DATA_DIR / "pca_component_scores.csv"
PCA_LEAKAGE_AUDIT_PATH = DATA_DIR / "pca_loading_leakage_audit.csv"

from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


def infer_pca_source_context(source_columns):
    source = str(source_columns).split(",")[0].strip()
    prefix_contexts = [
        ("OverVa", 1, "before target Q14", "no target-branch leakage"),
        ("Heshbon", 6, "checking-account block before target Q14", "no target-branch leakage"),
        ("KartisAsh", 8, "credit-card block before target Q14", "no target-branch leakage"),
        ("Halvaa", 13, "loan block immediately before target Q14", "related debt proxy but not target-derived"),
        ("Hovot", 14, "target/debt-detail branch", "excluded leakage source; should not appear"),
        ("Hashkaot", 16, "investment block after target Q14", "post-target questionnaire block; not Hovot-derived"),
        ("Bituhim", 18, "insurance block after target Q14", "post-target questionnaire block; not Hovot-derived"),
        ("NosCas", 20, "knowledge block after target Q14", "post-target questionnaire block; not Hovot-derived"),
        ("NoseCaspi", 25, "attitude block after target Q14", "post-target questionnaire block; not Hovot-derived"),
        ("Mekorot", 27, "information-source interest block after target Q14", "post-target questionnaire block; not Hovot-derived"),
    ]
    for prefix, question, context, note in prefix_contexts:
        if source.startswith(prefix):
            return question, context, prefix == "Hovot", note
    return np.nan, "profile/background or aggregate source", False, "not target-branch leakage"


def run_pca_analysis_sklearn(n_components=4, top_loading_count=30):
    data = pd.read_csv(LABELED_PATH)
    feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
    feature_names = [c for c in data.columns if c not in {"SerialNumber", "y_debt", "sample_weight"}]
    X_raw = data[feature_names].to_numpy(dtype=float)
    y = data["y_debt"].to_numpy(dtype=int)

    scaler = StandardScaler()
    X_scaled_all = scaler.fit_transform(X_raw)
    non_constant = scaler.scale_ > 1e-12
    X = X_scaled_all[:, non_constant]
    kept_features = np.array(feature_names, dtype=object)[non_constant]

    pca = PCA(n_components=n_components)
    scores = pca.fit_transform(X)
    source_lookup = feature_dictionary.drop_duplicates("feature_name").set_index("feature_name")

    explained = pd.DataFrame({
        "component": [f"PC{i + 1}" for i in range(n_components)],
        "eigenvalue": pca.explained_variance_,
        "explained_variance_ratio": pca.explained_variance_ratio_,
        "cumulative_explained_variance_ratio": np.cumsum(pca.explained_variance_ratio_),
    })

    baseline = float(y.mean())
    pr_rows, loading_rows, audit_rows = [], [], []
    score_output = pd.DataFrame({"SerialNumber": data["SerialNumber"], "y_debt": y})

    for i in range(n_components):
        pc = f"PC{i + 1}"
        alpha = scores[:, i]
        pr_alpha = average_precision_score(y, alpha)
        pr_neg = average_precision_score(y, -alpha)
        best_orientation = "alpha" if pr_alpha >= pr_neg else "-alpha"
        best_pr = max(pr_alpha, pr_neg)
        score_output[f"{pc}_score_raw"] = alpha
        score_output[f"{pc}_score_oriented"] = alpha if best_orientation == "alpha" else -alpha
        pr_rows.append({
            "component": pc,
            "pr_auc_alpha": pr_alpha,
            "pr_auc_negative_alpha": pr_neg,
            "best_orientation": best_orientation,
            "best_pr_auc": best_pr,
            "baseline_positive_rate": baseline,
            "best_pr_auc_minus_baseline": best_pr - baseline,
        })

        loadings = pca.components_[i]
        order = np.argsort(-np.abs(loadings))
        for rank, j in enumerate(order[:top_loading_count], start=1):
            feature = kept_features[j]
            row = {
                "component": pc,
                "rank": rank,
                "feature_name": feature,
                "abs_loading": abs(float(loadings[j])),
                "signed_loading": float(loadings[j]),
                "source_columns": source_lookup["source_columns"].get(feature, ""),
                "feature_type": source_lookup["feature_type"].get(feature, ""),
                "transformation": source_lookup["transformation"].get(feature, ""),
            }
            loading_rows.append(row)
            question, context, target_branch, note = infer_pca_source_context(row["source_columns"])
            audit_rows.append({**row, "approx_question": question, "question_context": context, "target_branch_source": target_branch, "leakage_audit_note": note})

        plt.figure(figsize=(12, 6))
        plt.plot(np.abs(loadings[order]), linewidth=1.5)
        plt.title(f"{pc}: absolute PCA loadings, descending")
        plt.xlabel("features sorted by |loading|")
        plt.ylabel("absolute loading")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / f"pca_{pc.lower()}_loadings.png", dpi=140)
        plt.close()

    loadings_df = pd.DataFrame(loading_rows)
    pr_auc_df = pd.DataFrame(pr_rows)
    leakage_audit = pd.DataFrame(audit_rows)
    safe_to_csv(explained, PCA_EXPLAINED_PATH)
    safe_to_csv(loadings_df, PCA_LOADINGS_PATH)
    safe_to_csv(pr_auc_df, PCA_PR_AUC_PATH)
    safe_to_csv(score_output, PCA_SCORES_PATH)
    safe_to_csv(leakage_audit, PCA_LEAKAGE_AUDIT_PATH)

    print("PCA completed with sklearn PCA")
    print(explained.to_string(index=False))
    print(pr_auc_df.to_string(index=False))
    print("Target-branch rows among top loadings:", int(leakage_audit["target_branch_source"].sum()))
    return {"explained_variance": explained, "pr_auc": pr_auc_df, "top_loadings": loadings_df, "scores": score_output, "leakage_audit": leakage_audit}


In [6]:
pca_results = run_pca_analysis_sklearn()
pca_results['pr_auc']


PCA completed with sklearn PCA
component  eigenvalue  explained_variance_ratio  cumulative_explained_variance_ratio
      PC1   29.929476                  0.071030                             0.071030
      PC2   23.159321                  0.054963                             0.125992
      PC3   14.200669                  0.033702                             0.159694
      PC4   13.572489                  0.032211                             0.191905
component  pr_auc_alpha  pr_auc_negative_alpha best_orientation  best_pr_auc  baseline_positive_rate  best_pr_auc_minus_baseline
      PC1      0.232127               0.175722            alpha     0.232127                 0.19948                    0.032648
      PC2      0.181387               0.229194           -alpha     0.229194                 0.19948                    0.029714
      PC3      0.155226               0.281849           -alpha     0.281849                 0.19948                    0.082369
      PC4      0.149610     

,component,pr_auc_alpha,pr_auc_negative_alpha,best_orientation,best_pr_auc,baseline_positive_rate,best_pr_auc_minus_baseline
0,PC1,0.232127,0.175722,alpha,0.232127,0.19948,0.032648
1,PC2,0.181387,0.229194,-alpha,0.229194,0.19948,0.029714
2,PC3,0.155226,0.281849,-alpha,0.281849,0.19948,0.082369
3,PC4,0.149610,0.334319,-alpha,0.334319,0.19948,0.134839


### Q4 - Prediction Models


In [7]:
DATA_DIR = SUPERVISED_DIR
SPLIT_PATH = DATA_DIR / "model_split_assignments.csv"
CV_RESULTS_PATH = DATA_DIR / "model_cv_results_part4.csv"
PERFORMANCE_PATH = DATA_DIR / "model_performance_part4.csv"
PREDICTIONS_PATH = DATA_DIR / "model_predictions_part4.csv"
LASSO_TOP15_PART4_PATH = DATA_DIR / "model_lasso_top15_features_part4.csv"
F1_PLOT_PNG_PATH = PLOT_DIR / "part4_f1_scores.png"

from sklearn.base import BaseEstimator, clone
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


SEED = 42
THRESHOLDS = np.round(np.arange(0.20, 0.61, 0.05), 2)


class ThresholdClassifier(BaseEstimator):
    def __init__(self, estimator=None, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(X, y)
        return self

    def predict(self, X):
        if hasattr(self.estimator_, "predict_proba"):
            scores = self.estimator_.predict_proba(X)[:, 1]
        else:
            raw = self.estimator_.decision_function(X)
            scores = 1 / (1 + np.exp(-raw))
        return (scores >= self.threshold).astype(int)

    def predict_proba(self, X):
        if hasattr(self.estimator_, "predict_proba"):
            return self.estimator_.predict_proba(X)
        raw = self.estimator_.decision_function(X)
        scores = 1 / (1 + np.exp(-raw))
        return np.column_stack([1 - scores, scores])


def build_split_and_features():
    data = pd.read_csv(LABELED_PATH)
    feature_names = [c for c in data.columns if c not in {"SerialNumber", "y_debt", "sample_weight"}]
    X = data[feature_names]
    y = data["y_debt"].astype(int)

    train_idx, test_idx = train_test_split(
        np.arange(len(data)),
        test_size=1/3,
        random_state=SEED,
        stratify=y,
    )
    split = pd.DataFrame({"SerialNumber": data["SerialNumber"], "y_debt": y, "split": "test"})
    split.loc[train_idx, "split"] = "train"

    assert len(train_idx) == 768 and len(test_idx) == 385
    safe_to_csv(split, SPLIT_PATH)
    return data, X, y, np.sort(train_idx), np.sort(test_idx), feature_names


def select_top15_from_training_lasso(feature_names, X_train, y_train):
    feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
    source_lookup = feature_dictionary.drop_duplicates("feature_name").set_index("feature_name")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train.to_numpy(dtype=float))
    non_constant = scaler.var_ > 1e-12
    kept_features = np.array(feature_names, dtype=object)[non_constant]
    X_path = X_scaled[:, non_constant]

    C_values = np.geomspace(0.05, 30.0, 35)
    coefs = []
    for C in C_values:
        model = LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=float(C),
            max_iter=5000,
            random_state=SEED,
        )
        model.fit(X_path, y_train)
        coefs.append(model.coef_.ravel())

    coef_path = np.vstack(coefs)
    active = np.abs(coef_path) > 1e-10
    entry_steps = np.where(active.any(axis=0), active.argmax(axis=0) + 1, np.inf)
    max_abs = np.abs(coef_path).max(axis=0)

    rows = []
    for j, feature in enumerate(kept_features):
        if max_abs[j] <= 0 or not np.isfinite(entry_steps[j]):
            continue
        rows.append({
            "feature_name": feature,
            "entry_step": int(entry_steps[j]),
            "entry_C": float(C_values[int(entry_steps[j]) - 1]),
            "max_abs_standardized_coef": float(max_abs[j]),
            "final_standardized_coef": float(coef_path[-1, j]),
            "source_columns": source_lookup["source_columns"].get(feature, ""),
            "feature_type": source_lookup["feature_type"].get(feature, ""),
            "transformation": source_lookup["transformation"].get(feature, ""),
        })

    ranked = pd.DataFrame(rows).sort_values(
        ["entry_step", "max_abs_standardized_coef"], ascending=[True, False]
    ).reset_index(drop=True)
    top15 = ranked.head(15).copy()
    top15.insert(0, "rank", np.arange(1, len(top15) + 1))
    safe_to_csv(top15, LASSO_TOP15_PART4_PATH)
    return top15["feature_name"].tolist()


def fit_grid(name, estimator, param_grid, X_train, y_train):
    grid = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        scoring="f1",
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED),
        n_jobs=-1,
        refit=True,
        return_train_score=True,
    )
    grid.fit(X_train, y_train)
    rows = pd.DataFrame(grid.cv_results_)
    rows["model"] = name
    return grid, rows


def run_part4_models_sklearn():
    data, X, y, train_idx, test_idx, feature_names = build_split_and_features()
    feature_sets = {
        "lasso_top15": select_top15_from_training_lasso(feature_names, X.iloc[train_idx], y.iloc[train_idx]),
    }
    model_specs = {
        "logistic_l2": (
            Pipeline([("scale", StandardScaler()), ("clf", ThresholdClassifier(LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000, random_state=SEED)))]),
            {"clf__estimator__C": [0.001, 0.01, 0.1, 1.0, 10.0], "clf__threshold": THRESHOLDS},
        ),
        "knn": (
            Pipeline([("scale", StandardScaler()), ("clf", ThresholdClassifier(KNeighborsClassifier(metric="euclidean")))]),
            {"clf__estimator__n_neighbors": [3, 5, 9, 15, 25, 35], "clf__threshold": THRESHOLDS},
        ),
        "kernel_logistic_rbf": (
            Pipeline([("scale", StandardScaler()), ("kernel", Nystroem(kernel="rbf", random_state=SEED, n_components=200)), ("clf", ThresholdClassifier(LogisticRegression(penalty="l2", solver="liblinear", max_iter=5000, random_state=SEED)))]),
            {"kernel__gamma": [0.005, 0.01, 0.02, 0.05], "clf__estimator__C": [0.1, 1.0, 10.0], "clf__threshold": THRESHOLDS},
        ),
    }

    cv_rows, performance_rows = [], []
    predictions = data[["SerialNumber", "y_debt"]].copy()
    predictions["split"] = "test"
    predictions.loc[train_idx, "split"] = "train"

    for feature_set, cols in feature_sets.items():
        X_train, X_test = X.iloc[train_idx][cols], X.iloc[test_idx][cols]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        for model_name, (estimator, grid_params) in model_specs.items():
            grid, cv = fit_grid(model_name, estimator, grid_params, X_train, y_train)
            cv["feature_set"] = feature_set
            cv_rows.append(cv)
            train_pred = grid.predict(X_train)
            test_pred = grid.predict(X_test)
            train_score = grid.predict_proba(X_train)[:, 1]
            test_score = grid.predict_proba(X_test)[:, 1]
            prefix = f"{model_name}_{feature_set}"
            predictions[f"{prefix}_pred"] = np.nan
            predictions[f"{prefix}_score"] = np.nan
            predictions.loc[train_idx, f"{prefix}_pred"] = train_pred
            predictions.loc[test_idx, f"{prefix}_pred"] = test_pred
            predictions.loc[train_idx, f"{prefix}_score"] = train_score
            predictions.loc[test_idx, f"{prefix}_score"] = test_score

            performance_rows.append({
                "model": model_name,
                "feature_set": feature_set,
                "params": str(grid.best_params_),
                "mean_cv_f1": float(grid.best_score_),
                "train_f1": f1_score(y_train, train_pred),
                "test_f1": f1_score(y_test, test_pred),
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_raw_features": len(cols),
            })

    cv_results = pd.concat(cv_rows, ignore_index=True)
    performance = pd.DataFrame(performance_rows)
    safe_to_csv(cv_results, CV_RESULTS_PATH)
    safe_to_csv(performance, PERFORMANCE_PATH)
    safe_to_csv(predictions, PREDICTIONS_PATH)

    plot_df = performance.copy()
    x_pos = np.arange(len(plot_df))
    width = 0.35
    plt.figure(figsize=(11, 5))
    plt.bar(x_pos - width/2, plot_df["train_f1"], width, label="Train F1")
    plt.bar(x_pos + width/2, plot_df["test_f1"], width, label="Test F1")
    plt.xticks(x_pos, plot_df["model"], rotation=20, ha="right")
    plt.ylabel("F1")
    plt.title("Part 4 model F1 scores")
    plt.legend()
    plt.tight_layout()
    plt.savefig(F1_PLOT_PNG_PATH, dpi=140)
    plt.close()

    print(performance[["model", "feature_set", "params", "mean_cv_f1", "train_f1", "test_f1"]].to_string(index=False))
    return {"performance": performance, "predictions": predictions, "cv_results": cv_results}


In [8]:
part4_results = run_part4_models_sklearn()
part4_results['performance']


              model feature_set                                                                     params  mean_cv_f1  train_f1  test_f1
        logistic_l2 lasso_top15                         {'clf__estimator__C': 10.0, 'clf__threshold': 0.3}    0.598328  0.618750 0.623377
                knn lasso_top15                {'clf__estimator__n_neighbors': 35, 'clf__threshold': 0.25}    0.575959  0.587896 0.573099
kernel_logistic_rbf lasso_top15 {'clf__estimator__C': 10.0, 'clf__threshold': 0.3, 'kernel__gamma': 0.005}    0.599264  0.610063 0.622517


,model,feature_set,params,mean_cv_f1,train_f1,test_f1,n_train,n_test,n_raw_features
0,logistic_l2,lasso_top15,"{'clf__estimator__C': 10.0, 'clf__threshold': ...",0.598328,0.618750,0.623377,768,385,15
1,knn,lasso_top15,"{'clf__estimator__n_neighbors': 35, 'clf__thre...",0.575959,0.587896,0.573099,768,385,15
2,kernel_logistic_rbf,lasso_top15,"{'clf__estimator__C': 10.0, 'clf__threshold': ...",0.599264,0.610063,0.622517,768,385,15


### Q5 - Learner Diversity and Majority Vote


In [9]:
DATA_DIR = SUPERVISED_DIR
ERRORS_PATH = DATA_DIR / "learner_errors_part5.csv"
ERROR_MATRIX_PATH = DATA_DIR / "learner_error_correlation_part5.csv"
MAJORITY_SUMMARY_PATH = DATA_DIR / "majority_vote_summary_part5.csv"
MAJORITY_PREDICTIONS_PATH = DATA_DIR / "majority_vote_predictions_part5.csv"


def run_part5_learner_diversity():
    predictions = pd.read_csv(PREDICTIONS_PATH)
    performance = pd.read_csv(PERFORMANCE_PATH)
    learners = ["logistic_l2", "knn", "kernel_logistic_rbf"]
    pred_cols = [f"{learner}_lasso_top15_pred" for learner in learners]
    train_mask = predictions["split"].eq("train")
    test_mask = predictions["split"].eq("test")

    errors = predictions.loc[train_mask, ["SerialNumber", "y_debt"]].copy()
    for learner, col in zip(learners, pred_cols):
        errors[f"{learner}_error"] = (predictions.loc[train_mask, col].astype(int).to_numpy() != predictions.loc[train_mask, "y_debt"].astype(int).to_numpy()).astype(int)

    E = errors[[f"{learner}_error" for learner in learners]].to_numpy(dtype=float)
    E_tilde = (E - E.mean(axis=0)) / np.sqrt(((E - E.mean(axis=0)) ** 2).sum(axis=0))
    diversity = pd.DataFrame(E_tilde.T @ E_tilde, index=learners, columns=learners)
    diversity.index.name = "learner"

    majority_pred = (predictions[pred_cols].astype(int).sum(axis=1) >= 2).astype(int)
    summary_rows = []
    for learner, col in zip(learners, pred_cols):
        row = performance.loc[performance["model"].eq(learner)].iloc[0]
        summary_rows.append({"learner": learner, "train_f1": row["train_f1"], "test_f1": row["test_f1"], "combination": "base"})
    summary_rows.append({
        "learner": "majority_vote",
        "train_f1": f1_score_binary(predictions.loc[train_mask, "y_debt"], majority_pred[train_mask]),
        "test_f1": f1_score_binary(predictions.loc[test_mask, "y_debt"], majority_pred[test_mask]),
        "combination": "majority_vote",
    })
    summary = pd.DataFrame(summary_rows)
    best_base = summary.loc[summary["combination"].eq("base"), "test_f1"].max()
    summary["beats_best_base_test_f1"] = False
    summary.loc[summary["learner"].eq("majority_vote"), "beats_best_base_test_f1"] = summary.loc[summary["learner"].eq("majority_vote"), "test_f1"].iloc[0] > best_base

    out = predictions[["SerialNumber", "y_debt", "split"] + pred_cols].copy()
    out["majority_vote_pred"] = majority_pred
    out["majority_vote_error"] = (majority_pred != predictions["y_debt"].astype(int)).astype(int)

    safe_to_csv(errors, ERRORS_PATH)
    safe_to_csv(diversity.reset_index(), ERROR_MATRIX_PATH)
    safe_to_csv(summary, MAJORITY_SUMMARY_PATH)
    safe_to_csv(out, MAJORITY_PREDICTIONS_PATH)

    print(diversity.round(4).to_string())
    print(summary.to_string(index=False))
    return {"errors": errors, "diversity_matrix": diversity, "summary": summary, "predictions": out}


In [10]:
part5_results = run_part5_learner_diversity()
part5_results['summary']


                     logistic_l2     knn  kernel_logistic_rbf
learner                                                      
logistic_l2               1.0000  0.6798               0.9226
knn                       0.6798  1.0000               0.7083
kernel_logistic_rbf       0.9226  0.7083               1.0000
            learner  train_f1  test_f1   combination  beats_best_base_test_f1
        logistic_l2  0.618750 0.623377          base                    False
                knn  0.587896 0.573099          base                    False
kernel_logistic_rbf  0.610063 0.622517          base                    False
      majority_vote  0.619195 0.618421 majority_vote                    False


,learner,train_f1,test_f1,combination,beats_best_base_test_f1
0,logistic_l2,0.618750,0.623377,base,False
1,knn,0.587896,0.573099,base,False
2,kernel_logistic_rbf,0.610063,0.622517,base,False
3,majority_vote,0.619195,0.618421,majority_vote,False
